# Prototype

## Channel Model – Key Relations

$th(x) = gpc \left( 1 - \exp(-c Q(x) t_p) \right)$  

$Q(x) = Q_0 \exp(-x/x_s)$  

$x_s \propto \sqrt{T/p_A}$

In [17]:
import torch
import torch.nn as nn

from torchdiffeq import odeint
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Channel Profile

In [18]:
N0 = 6.22*10**23
k = 1.38*10**-23 # Boltzmann constant
R = 8.314 # J/K.mol

In [19]:
class channelModel():

    def __init__(self):

        self.M = 101.96/1000 # kg/mol (molar mass of formula unit of film)
        self.rho = 3.99*1000 #density of film material

        self.b_film = 2 # number of metal atoms in the formula unit for the film
        self.b_a = 1 # number of metal atoms in the reactant, i.e. TMA
        self.c = 0.01 # c = sticking probability
        #self.Pd = 0 # Pd = Desorption probability
        self.da = 591e-12 #tma #d_estimate()
        self.db = 418e-12 #water #d_estimate()
        # Molar Mass
        self.MA = 72.09e-3 #kg/mol
        self.MB = 18e-3

        # Channel Geometry
        self.H = 0.2e-6
        self.W = 0.1e-3

        self.gpc = 106 * 10**-12 # gpc (m)
        self.K = 100 # equilibrium constant for langmuir model
        # or K = cQ/(qP_d)
        # But they seem to generally give K as its own value
        # c is used for calculating the delimiter instead

        self.T = 300 + 273 # Temperature
        self.t_p = 0.1 #pulse time

        self.pA = 100 #*1e-3 # Partial Pressure of reactant A (limiting diffusion)
        #self.pA0 = self.pA
        self.pB = 300 # partial pressure of reactant B

    def calc_hydro_diameter(self):
        self.h = 2/(1/self.H + 1/self.W)
    def calc_adsorption_density(self):
        self.q = (self.b_film/self.b_a) * (self.rho*self.gpc/self.M) * N0 #TMA q (adsorption density)
    def collision_rate(self):
        self.Q = N0/np.sqrt(2*np.pi* self.M * R * self.T)# Q: collision rate at unit pressure

    def calc_za(self):
        #find collision rate of a; (A+B) collisions and A+A collisions
        #molecular diameter, molar mass, partial pressure
        self.za = (np.pi/4*((self.da+self.db)**2)*np.sqrt(8*R*self.T/np.pi*(1/self.MA + 1/self.MB)) * self.pB + \
                   np.pi * self.da**2 * np.sqrt(16*R*self.T/(np.pi*self.MA))*self.pA) /(R*self.T)
    def calc_Deff(self):
        z_a = self.za #(self.da, self.db, self.MA, self.MB, self.pA, self.pB)
        va = np.sqrt(8*R*self.T/(np.pi * self.MA))
        Da = 3*np.pi/16 * va**2/z_a
        Dkn = self.h*np.sqrt(8*R*self.T/(9*np.pi*self.MA))
        self.Deff = 1/(1/Da + 1/Dkn)

    def approx(self, x, last_theta):

        D = self.pA*self.Deff*self.H/(self.q*k*self.T*(1-np.log(self.K*self.pA+1)/(self.K*self.pA)))
        xs = np.sqrt(D*self.t_p)
        delim = np.sqrt((self.h*N0*self.Deff)/(4*R*self.T*self.c*self.Q))
        xt = xs-delim #np.max(0.0, xs-delim)
        if xt < 0:
            xt = 0
        pt = self.pA * (1-xt/xs)
        pA = self.pA * (1-x/xs)
        pA[x>xt] = pt*np.exp(-(x[x>xt]-xt)/(xs-xt))

        theta = (self.K*pA)/(1+self.K*pA)

        next_theta = last_theta + theta

        thickness = self.gpc*next_theta

        #print('xt: ', xt)

        return thickness, next_theta, {'theta':theta, 'pA': pA}

    def run(self, steps, initial_x):

        results = []#pd.DataFrame()

        #Assuming no temperature change, these don't need recalculation
        self.collision_rate()
        self.calc_adsorption_density()
        self.calc_za()
        last_theta = initial_x.copy()

        original_H = self.H

        for step in range(steps):
            self.calc_hydro_diameter()
            self.calc_Deff()
            thickness, last_theta, intermediates = self.approx(x, last_theta)
            results.append(pd.DataFrame({'thickness': thickness} | intermediates))
            self.H = self.H - 2 * self.gpc

        self.H = original_H

        results = pd.concat(results, keys=range(steps))
        return results




In [20]:
model = channelModel()
#model.pA *= 2
model.t_p*=10
model.H = 0.5* 0.2e-6
model.W = 1000*0.2e-6

model.pA = max(model.pA, 1e-3)
model.K = max(model.K, 1e-6)

In [21]:
import numpy as np
import pandas as pd

# ===== 설정 =====
N = 50                 # step 수
gpc_nominal = 1.2e-10  # 기존 constant 값
noise_level = 0.1      # 10% noise (0.1 = 10%)

# ===== noise 생성 =====
# multiplicative noise (추천)
noise = 1 + noise_level * np.random.randn(N)

gpc_values = gpc_nominal * noise

# 음수 방지
gpc_values = np.clip(gpc_values, 0, None)

# ===== dataframe =====
df = pd.DataFrame({
    "step": np.arange(N),
    "gpc": gpc_values
})

# ===== 저장 =====
df.to_csv("gpc_data.csv", index=False)

print("gpc_data.csv 생성 완료")
df.head()

gpc_data.csv 생성 완료


,step,gpc
0,0,1.221888e-10
1,1,1.233392e-10
2,2,1.093659e-10
3,3,1.216749e-10
4,4,1.274758e-10


In [22]:
import pandas as pd

gpc_df = pd.read_csv("gpc_data.csv")

def get_gpc(k):
    if k < len(gpc_df):
        return gpc_df.iloc[k]["gpc"]
    else:
        return gpc_df.iloc[-1]["gpc"]

In [ ]:
# ============================================================
# Cell 1: 유틸리티 + 모델 확장
# ============================================================

import numpy as np
import pandas as pd
import copy
from scipy.optimize import minimize

# ----- gpc CSV reader -----
def get_gpc(k, path="gpc_data.csv"):
    gpc_df = pd.read_csv(path)
    if k < len(gpc_df):
        return float(gpc_df.iloc[k]["gpc"])
    return float(gpc_df.iloc[-1]["gpc"])


# ----- channelModel 확장 -----
class channelModelMPC(channelModel):

    def approx(self, x, last_theta):
        D = (self.pA * self.Deff * self.H /
             (self.q * k * self.T *
              (1 - np.log(self.K * self.pA + 1) / (self.K * self.pA))))
        xs    = np.sqrt(D * self.t_p)
        delim = np.sqrt((self.h * N0 * self.Deff) /
                        (4 * R * self.T * self.c * self.Q))
        xt = max(xs - delim, 0.0)

        pt = self.pA * (1 - xt / xs) if xs > 0 else 0.0
        pA_profile = self.pA * (1 - x / xs)
        if xs - xt > 0:
            pA_profile[x > xt] = pt * np.exp(-(x[x > xt] - xt) / (xs - xt))
        else:
            pA_profile[x > xt] = 0.0
        pA_profile = np.clip(pA_profile, 0, None)

        theta      = (self.K * pA_profile) / (1 + self.K * pA_profile)
        next_theta = last_theta + theta
        thickness  = self.gpc * next_theta

        return thickness, next_theta, {
            'pA'   : pA_profile,
            'theta': theta,
            'xs'   : xs,
            'xt'   : xt,
            'delim': delim,
        }

    def prepare(self):
        self.collision_rate()
        self.calc_adsorption_density()
        self.calc_za()
        self.calc_hydro_diameter()
        self.calc_Deff()


# ============================================================
# Cell 2: x_half 계산 (pA profile 기준)
# ============================================================

def find_x_half_pA(pA_profile, x):
    """
    pA(x) 가 입구값의 50% 가 되는 채널 위치.
    K=100 → theta≈1 → thickness flat → pA profile 이 유일한 공간 지표.
    """
    pA_inlet = pA_profile[0]
    if pA_inlet <= 0:
        return 0.0
    pA_half = 0.5 * pA_inlet
    idx = np.where(pA_profile <= pA_half)[0]
    if len(idx) == 0:
        return x[-1]
    i = idx[0]
    if i == 0:
        return x[0]
    slope = (pA_profile[i] - pA_profile[i-1]) / (x[i] - x[i-1])
    return float(x[i-1] + (pA_half - pA_profile[i-1]) / slope)


# ============================================================
# Cell 3: 단일 스텝 평가
# ============================================================

def evaluate_step(pA_val, tp_val, model, x, last_theta):
    m = copy.copy(model)          # model 상태 보호 (optimizer iteration 간 오염 방지)
    m.pA  = max(pA_val, 1e-3)
    m.t_p = tp_val
    m.prepare()
    thickness, next_theta, info = m.approx(x, last_theta)
    x_half = find_x_half_pA(info['pA'], x)
    return thickness, next_theta, x_half, info


# ============================================================
# Cell 4: MPC 1-step optimizer
# ============================================================

def mpc_step(model, x, last_theta, l_0,
             pA_bounds, tp_bounds, alpha, beta):
    """
    min   (pA/pA_max)*(t_p/tp_max)  +  alpha*(pA/pA_max)  +  beta*(t_p/tp_max)
    s.t.  x_half(pA, t_p) >= l_0

    각 항의 의미 (모두 정규화):
      pA*t_p 항  : 반응물 총 사용량
      alpha*pA   : 압력 자체 비용 (가스 소모, 장비 부하)
      beta*t_p   : 시간 자체 비용 (throughput)

    alpha 크면 pA 낮추고 t_p 로 보상
    beta  크면 t_p 낮추고 pA 로 보상
    → pA 와 t_p 가 objective 에서 독립적으로 등장 → 유일한 최적해
    """
    pA_max = pA_bounds[1]
    tp_max = tp_bounds[1]

    def objective(u):
        pA_val, tp_val = u
        
        # normalize
        tp_n = tp_val / tp_max
        dP_n = (pA_val - pA_prev) / (pA_max - pA_min)
        
        return w_t * tp_n + w_P * (dP_n**2)

    # 최적화는 현재 사이클 단독 기준으로 평가
    # x_half 는 pA profile 기반이라 last_theta 와 무관하나
    # approx() 내부 혼선 방지를 위해 zeros 명시
    _zeros = np.zeros_like(x)

    def con_x_half(u):
        pA_val, tp_val = u
        _, _, x_half, _ = evaluate_step(pA_val, tp_val, model, x, _zeros)
        return x_half - l_0   # >= 0

    u0     = [model.pA, model.t_p]
    bounds = [pA_bounds, tp_bounds]
    constr = {'type': 'ineq', 'fun': con_x_half}

    result = minimize(
        objective, u0,
        method='SLSQP',
        bounds=bounds,
        constraints=constr,
        options={'ftol': 1e-10, 'maxiter': 500}
    )

    if not result.success:
        print(f"  [WARN] {result.message}")

    pA_opt, tp_opt = result.x
    return pA_opt, tp_opt, result


# ============================================================
# Cell 5: MPC 루프
# ============================================================

def run_mpc(x, steps, l_0,
            T_fixed, pA_bounds, tp_bounds, alpha, beta,
            gpc_csv = "gpc_data.csv"):
    """
    MPC 메인 루프.

    Objective (정규화):
      min  pA*t_p  +  alpha*pA  +  beta*t_p   (모두 /max 정규화)

    alpha > beta : pA 낮추는 방향 선호
    beta  > alpha: t_p 낮추는 방향 선호
    alpha = beta : 균등 tradeoff
    """
    mpc_model     = channelModelMPC()
    mpc_model.T   = T_fixed
    mpc_model.pA  = np.mean(pA_bounds)
    mpc_model.t_p = np.mean(tp_bounds)

    last_theta = np.zeros_like(x)
    history    = []

    for k in range(steps):

        # ── 1. gpc 업데이트 ──
        mpc_model.gpc = get_gpc(k, gpc_csv)

        # ── 2. 최적 (pA, t_p) 계산 ──
        pA_opt, tp_opt, result = mpc_step(
            mpc_model, x, last_theta, l_0,
            pA_bounds=pA_bounds,
            tp_bounds=tp_bounds,
            alpha=alpha,
            beta=beta,
        )

        # ── 3. 적용 → 다음 상태 ──
        thickness, last_theta, x_half, info = evaluate_step(
            pA_opt, tp_opt, mpc_model, x, last_theta
        )

        pA_max = pA_bounds[1]
        tp_max = tp_bounds[1]
        pA_n   = pA_opt / pA_max
        tp_n   = tp_opt / tp_max
        cost   = pA_n * tp_n + alpha * pA_n + beta * tp_n
        ok     = x_half >= l_0 - 1e-9   # floating point tolerance

        history.append({
            'step'          : k,
            'gpc'           : mpc_model.gpc,
            'pA'            : pA_opt,
            't_p'           : tp_opt,
            'cost'          : cost,
            'pA_tp'         : pA_opt * tp_opt,
            'x_half'        : x_half,
            'l_0'           : l_0,
            'constraint_ok' : ok,
            'xs'            : info['xs'],
            'thickness_in'  : thickness[0],
            'thickness_tip' : thickness[-1],
        })

        # warm-start
        mpc_model.pA  = pA_opt
        mpc_model.t_p = tp_opt

        print(f"Step {k:3d} | gpc={mpc_model.gpc:.3e} | "
              f"pA={pA_opt:.2f}Pa | t_p={tp_opt:.4f}s | "
              f"pA·t_p={pA_opt*tp_opt:.3f} | "
              f"x_half={x_half*1e3:.4f}mm | "
              f"{'✓' if ok else '✗ VIOLATION'}")

    return pd.DataFrame(history)


# ============================================================
# Cell 6: 실행 & 시각화
# ============================================================

x         = np.linspace(0, 1e-3, 500)
l_0       = 0.2e-3
T_fixed   = 400 + 273
pA_bounds = (10.0, 500.0)
tp_bounds = (0.01, 10.0)

# alpha, beta 설정
# alpha=1, beta=1 : pA 와 t_p 동등 페널티  → 균형잡힌 tradeoff
# alpha>beta      : pA 낮추는 방향 선호    → 가스 절약 우선
# beta>alpha      : t_p 낮추는 방향 선호   → throughput 우선
alpha = 0.0   # pA 페널티 약간 높게: pA 낮추되 t_p bound 에 안 붙는 수준
beta  = 0.0
w_t = 1.0
w_P = 1.0   

# ── feasibility 사전 확인 ──
_check     = channelModelMPC()
_check.T   = T_fixed
_check.gpc = 1.2e-10
_, _, xh_max, _ = evaluate_step(
    pA_bounds[1], tp_bounds[1], _check, x, np.zeros_like(x)
)
print(f"[Feasibility] pA={pA_bounds[1]}Pa, t_p={tp_bounds[1]}s "
      f"-> x_half={xh_max*1e3:.4f}mm  (l_0={l_0*1e3:.3f}mm)")
print("  ✓ Feasible" if xh_max >= l_0 else
      f"  ⚠ INFEASIBLE: l_0을 {xh_max*1e3:.4f}mm 이하로 낮추거나 bounds를 넓히세요.")

# ── MPC 실행 ──
history_df = run_mpc(
    x         = x,
    steps     = 50,
    l_0       = l_0,
    T_fixed   = T_fixed,
    pA_bounds = pA_bounds,
    tp_bounds = tp_bounds,
    alpha     = alpha,
    beta      = beta,
    gpc_csv   = "gpc_data.csv",
    
)


# ── cost 컬럼 추가 ──
pA_max_val = pA_bounds[1]
tp_max_val = tp_bounds[1]
history_df['cost'] = (
    (history_df['pA'] / pA_max_val) * (history_df['t_p'] / tp_max_val)
    + alpha * (history_df['pA'] / pA_max_val)
    + beta  * (history_df['t_p'] / tp_max_val)
)

# ── Pareto optimal 포인트 추출 (pA, t_p 두 목표) ──
def is_pareto(costs):
    n = len(costs)
    mask = np.ones(n, dtype=bool)
    for i in range(n):
        if mask[i]:
            dominated = (
                np.all(costs <= costs[i], axis=1) &
                np.any(costs <  costs[i], axis=1)
            )
            dominated[i] = False
            if dominated.any():
                mask[i] = False
    return mask

pareto_costs = history_df[['pA', 't_p']].values
pareto_mask  = is_pareto(pareto_costs)

# ── Fig 1: 시계열 ──
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(
    f"MPC: min  pA·t_p  +  α·pA  +  β·t_p   (α={alpha}, β={beta})\n"
    f"s.t.  x_half ≥ l₀={l_0*1e3:.1f}mm",
    fontsize=12
)
steps_arr = history_df['step']

axes[0,0].plot(steps_arr, history_df['pA'], color='steelblue')
axes[0,0].set_title("Pressure pA (Pa)")
axes[0,0].set_xlabel("Step")

axes[0,1].plot(steps_arr, history_df['t_p'], color='darkorange')
axes[0,1].set_title("Pulse Time t_p (s)")
axes[0,1].set_xlabel("Step")

axes[0,2].plot(steps_arr, history_df['pA_tp'], color='crimson')
axes[0,2].set_title("pA × t_p  (Pa·s)")
axes[0,2].set_xlabel("Step")

axes[1,0].plot(steps_arr, history_df['x_half']*1e3, label="x_half (mm)", color='green')
axes[1,0].axhline(l_0*1e3, linestyle='--', color='red', label="l0")
axes[1,0].set_title("x_half vs l0")
axes[1,0].set_xlabel("Step")
axes[1,0].set_ylabel("mm")
axes[1,0].legend()

axes[1,1].plot(steps_arr, history_df['gpc']*1e12, color='purple')
axes[1,1].set_title("gpc per step (pm)")
axes[1,1].set_xlabel("Step")

# 총 cost 시계열
axes[1,2].plot(steps_arr, history_df['cost'], color='black', linewidth=1.5)
axes[1,2].set_title("Total Cost (normalized)")
axes[1,2].set_xlabel("Step")

plt.tight_layout()
plt.savefig("mpc_result.png", dpi=150)
plt.show()

# ── Fig 2: Pareto Front ──
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
fig2.suptitle("Pareto Front: pA vs t_p  (constraint: x_half ≥ l₀)", fontsize=12)

# 왼쪽: color = total cost
sc = axes2[0].scatter(
    history_df['pA'], history_df['t_p'],
    c=history_df['cost'], cmap='RdYlGn_r', s=50, zorder=2, label='all steps'
)
plt.colorbar(sc, ax=axes2[0], label='Total Cost')
axes2[0].scatter(
    history_df.loc[pareto_mask, 'pA'],
    history_df.loc[pareto_mask, 't_p'],
    edgecolors='black', facecolors='none', s=150, linewidths=2,
    zorder=3, label='Pareto optimal'
)
pf = history_df[pareto_mask].sort_values('pA')
axes2[0].plot(pf['pA'], pf['t_p'], 'k--', linewidth=1, alpha=0.6)
axes2[0].set_xlabel("pA (Pa)")
axes2[0].set_ylabel("t_p (s)")
axes2[0].set_title("pA vs t_p  (color=cost)\n○ = Pareto optimal")
axes2[0].legend(fontsize=8)

# 오른쪽: color = gpc (gpc 변화에 따른 최적해 이동)
sc2 = axes2[1].scatter(
    history_df['pA'], history_df['t_p'],
    c=history_df['gpc']*1e12, cmap='viridis', s=50, zorder=2
)
plt.colorbar(sc2, ax=axes2[1], label='gpc (pm)')
axes2[1].scatter(
    history_df.loc[pareto_mask, 'pA'],
    history_df.loc[pareto_mask, 't_p'],
    edgecolors='black', facecolors='none', s=150, linewidths=2,
    zorder=3, label='Pareto optimal'
)
axes2[1].set_xlabel("pA (Pa)")
axes2[1].set_ylabel("t_p (s)")
axes2[1].set_title("pA vs t_p  (color=gpc)\ngpc 변화에 따른 최적해 이동")
axes2[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("mpc_pareto.png", dpi=150)
plt.show()

print("\n=== Summary ===")
print(history_df[['step','gpc','pA','t_p','pA_tp','cost',
                   'x_half','constraint_ok']].to_string())


[Feasibility] pA=500.0Pa, t_p=10.0s -> x_half=0.5222mm  (l_0=0.200mm)
  ✓ Feasible


NameError: name 'pA_prev' is not defined

In [ ]:
model_diag = channelModelMPC()
model_diag.T   = 400 + 273
model_diag.gpc = 1.2e-10

# pA*t_p = 200 으로 고정하고 pA만 바꿔보기
target = 200.0
for pA in [50, 100, 150, 200]:
    tp = target / pA
    _, _, xh, info = evaluate_step(pA, tp, model_diag, x, np.zeros_like(x))
    print(f"pA={pA:5.0f} t_p={tp:.3f} | xs={info['xs']*1e3:.4f}mm | "
          f"delim={info['delim']*1e3:.4f}mm | x_half={xh*1e3:.4f}mm")

pA=   50 t_p=4.000 | xs=0.2090mm | delim=0.0025mm | x_half=0.1045mm
pA=  100 t_p=2.000 | xs=0.2089mm | delim=0.0025mm | x_half=0.1045mm
pA=  150 t_p=1.333 | xs=0.2089mm | delim=0.0025mm | x_half=0.1045mm
pA=  200 t_p=1.000 | xs=0.2089mm | delim=0.0025mm | x_half=0.1044mm
